# Evaluation breakdown

This notebook reads saved predictions. It does not load a model and does not use a GPU.
It joins the prediction rows to the stego rows in `labels.csv`.

## 1. Load predictions and labels

Set `PRED_PATH` at the top or set the `PRED_PATH` environment variable. The default is the file written by notebook 01.
The notebook also accepts `RUN_NAME` when `PRED_PATH` is not set.

In [ ]:
from pathlib import Path
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "Datasets").exists() and Path("/work/Datasets").exists():
    ROOT = Path("/work")

RUN_NAME = os.environ.get("RUN_NAME", "baseline_kv_xunet")
DEFAULT_PRED_PATH = Path("checkpoints") / RUN_NAME / "test_predictions.csv"
PRED_PATH = Path(os.environ.get("PRED_PATH", str(DEFAULT_PRED_PATH)))
if not PRED_PATH.is_absolute():
    PRED_PATH = ROOT / PRED_PATH

DATA_ROOT = ROOT / "Datasets" / "raise512-run1"
LABELS_PATH = DATA_ROOT / "labels.csv"
OUTPUT_DIR = PRED_PATH.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert PRED_PATH.exists(), f"Missing predictions file: {PRED_PATH}"
assert LABELS_PATH.exists(), f"Missing labels file: {LABELS_PATH}"

predictions = pd.read_csv(PRED_PATH, dtype={"cover_id": "string"})
labels = pd.read_csv(LABELS_PATH, dtype={"cover_id": "string"})

required_prediction_columns = {"cover_id", "label", "prob_stego"}
missing = required_prediction_columns - set(predictions.columns)
if missing:
    raise ValueError(f"Predictions are missing columns: {sorted(missing)}")
if "cover_id" not in labels or "label" not in labels:
    raise ValueError("Labels must contain cover_id and label")

def normalise_cover_id(values: pd.Series) -> pd.Series:
    return values.astype("string").str.strip().str.zfill(6)

predictions["cover_id"] = normalise_cover_id(predictions["cover_id"])
labels["cover_id"] = normalise_cover_id(labels["cover_id"])
predictions["label"] = pd.to_numeric(predictions["label"], errors="raise").astype(int)
labels["label"] = pd.to_numeric(labels["label"], errors="raise").astype(int)
predictions["prob_stego"] = pd.to_numeric(predictions["prob_stego"], errors="raise")
if not np.isfinite(predictions["prob_stego"].to_numpy(dtype=np.float64)).all():
    raise ValueError("prob_stego contains a non-finite value")
if set(predictions["label"].unique()) != {0, 1}:
    raise ValueError("Predictions must contain both labels 0 and 1")

extra_columns = [
    "stride",
    "start_random",
    "payload_bytes",
    "units_changed",
    "bpp_actual",
    "size_value",
]
setting_columns = ["bit_mode", "k", "method", "placement", "payload_kind", *extra_columns]
label_setting_columns = ["cover_id", *setting_columns]
missing_labels = set(label_setting_columns) - set(labels.columns)
if missing_labels:
    raise ValueError(f"Labels are missing columns: {sorted(missing_labels)}")

stego_labels = labels.loc[labels["label"] == 1, label_setting_columns].copy()
if stego_labels["cover_id"].duplicated().any():
    raise ValueError("labels.csv has more than one stego row for a cover_id")
frame = predictions.merge(
    stego_labels,
    on="cover_id",
    how="left",
    suffixes=("_prediction", "_labels"),
    validate="many_to_one",
)
for column in setting_columns:
    label_column = f"{column}_labels" if f"{column}_labels" in frame else column
    frame[column] = frame[label_column]
required_metadata_columns = [column for column in setting_columns if column != "stride"]
if frame[required_metadata_columns].isna().any().any():
    raise ValueError("A prediction row has no matching stego metadata in labels.csv")

print(f"Root: {ROOT}")
print(f"Predictions: {PRED_PATH}")
print(f"Prediction rows: {len(frame):,}")
print(f"Labels: {LABELS_PATH}")
print(f"Output folder: {OUTPUT_DIR}")

## 2. Overall result

`P_E` is the lowest average of false alarms and missed detections over all tied score thresholds.
The same global threshold is used for `P_FA` and `P_MD`.

In [ ]:
def _threshold_counts(labels_array: np.ndarray, scores_array: np.ndarray) -> tuple[np.ndarray, np.ndarray, int, int]:
    """Count positives and negatives at tied score thresholds."""
    labels_array = np.asarray(labels_array, dtype=np.int64)
    scores_array = np.asarray(scores_array, dtype=np.float64)
    if labels_array.size == 0:
        return np.array([], dtype=np.int64), np.array([], dtype=np.int64), 0, 0
    order = np.argsort(-scores_array, kind="mergesort")
    sorted_scores = scores_array[order]
    sorted_labels = labels_array[order]
    group_ends = np.flatnonzero(sorted_scores[1:] != sorted_scores[:-1]) + 1
    group_ends = np.r_[group_ends, len(sorted_scores)]
    cumulative_positive = np.cumsum(sorted_labels == 1, dtype=np.int64)
    cumulative_negative = np.cumsum(sorted_labels == 0, dtype=np.int64)
    return (
        cumulative_positive[group_ends - 1],
        cumulative_negative[group_ends - 1],
        int((labels_array == 1).sum()),
        int((labels_array == 0).sum()),
    )


def auc_numpy(labels_array: np.ndarray, scores_array: np.ndarray) -> float:
    """Compute ROC AUC with one sort and tied-score groups."""
    true_positive, false_positive, positives, negatives = _threshold_counts(labels_array, scores_array)
    if positives == 0 or negatives == 0:
        return 0.0
    denominator_positive = float(positives)
    denominator_negative = float(negatives)
    true_positive_rate = np.r_[0.0, true_positive / denominator_positive, true_positive[-1] / denominator_positive]
    false_positive_rate = np.r_[0.0, false_positive / denominator_negative, false_positive[-1] / denominator_negative]
    return float(np.trapezoid(true_positive_rate, false_positive_rate))


def probability_error(labels_array: np.ndarray, scores_array: np.ndarray) -> float:
    """Compute minimum P_E with one sort and tied-score groups."""
    true_positive, false_positive, positives, negatives = _threshold_counts(labels_array, scores_array)
    if positives == 0 and negatives == 0:
        return 0.0
    if negatives:
        false_alarm = np.r_[0.0, false_positive / negatives, false_positive[-1] / negatives]
    else:
        false_alarm = np.zeros(len(true_positive) + 2, dtype=np.float64)
    if positives:
        missed_detection = np.r_[1.0, (positives - true_positive) / positives, 0.0]
    else:
        missed_detection = np.zeros(len(true_positive) + 2, dtype=np.float64)
    return float(np.min((false_alarm + missed_detection) / 2.0))


def reference_auc(labels_array: np.ndarray, scores_array: np.ndarray) -> float:
    thresholds = np.r_[np.inf, np.sort(np.unique(scores_array))[::-1], -np.inf]
    positives = max(int((labels_array == 1).sum()), 1)
    negatives = max(int((labels_array == 0).sum()), 1)
    tpr = []
    fpr = []
    for threshold in thresholds:
        predicted = scores_array >= threshold
        tpr.append(float(((predicted == 1) & (labels_array == 1)).sum()) / positives)
        fpr.append(float(((predicted == 1) & (labels_array == 0)).sum()) / negatives)
    return float(np.trapezoid(np.asarray(tpr), np.asarray(fpr)))


def reference_probability_error(labels_array: np.ndarray, scores_array: np.ndarray) -> float:
    positives = labels_array == 1
    negatives = labels_array == 0
    thresholds = np.r_[np.inf, np.sort(np.unique(scores_array))[::-1], -np.inf]
    errors = []
    for threshold in thresholds:
        predicted = scores_array >= threshold
        p_fa = float(predicted[negatives].mean()) if negatives.any() else 0.0
        p_md = float((~predicted[positives]).mean()) if positives.any() else 0.0
        errors.append((p_fa + p_md) / 2.0)
    return float(min(errors))


check_rng = np.random.default_rng(123)
check_labels = check_rng.integers(0, 2, size=500, dtype=np.int64)
check_scores = np.round(check_rng.random(500), 2)
assert np.unique(check_scores).size < len(check_scores), "test array must contain ties"
fast_auc = auc_numpy(check_labels, check_scores)
reference_auc_value = reference_auc(check_labels, check_scores)
fast_p_e = probability_error(check_labels, check_scores)
reference_p_e = reference_probability_error(check_labels, check_scores)
assert np.isclose(fast_auc, reference_auc_value, atol=1e-12, rtol=0)
assert np.isclose(fast_p_e, reference_p_e, atol=1e-12, rtol=0)
print(f"AUC agrees within 1e-12: {np.isclose(fast_auc, reference_auc_value, atol=1e-12, rtol=0)}")
print(f"P_E agrees within 1e-12: {np.isclose(fast_p_e, reference_p_e, atol=1e-12, rtol=0)}")

labels_array = frame["label"].to_numpy(dtype=np.int64)
scores_array = frame["prob_stego"].to_numpy(dtype=np.float64)
thresholds = np.r_[np.inf, np.sort(np.unique(scores_array))[::-1], -np.inf]
threshold_errors = []
for threshold in thresholds:
    predicted = scores_array >= threshold
    p_fa = float(predicted[labels_array == 0].mean())
    p_md = float((~predicted[labels_array == 1]).mean())
    threshold_errors.append((p_fa + p_md) / 2.0)
best_threshold_index = int(np.argmin(threshold_errors))
GLOBAL_THRESHOLD = float(thresholds[best_threshold_index])
GLOBAL_P_E = probability_error(labels_array, scores_array)
global_predicted = scores_array >= GLOBAL_THRESHOLD
GLOBAL_P_FA = float(global_predicted[labels_array == 0].mean())
GLOBAL_P_MD = float((~global_predicted[labels_array == 1]).mean())
GLOBAL_DETECTION_RATE = float(global_predicted[labels_array == 1].mean())
assert np.isclose(GLOBAL_P_E, threshold_errors[best_threshold_index], atol=1e-12, rtol=0)
assert np.isclose(GLOBAL_DETECTION_RATE, 1.0 - GLOBAL_P_MD, atol=1e-12, rtol=0)

print(f"Images: {len(frame):,}; clean: {(labels_array == 0).sum():,}; stego: {(labels_array == 1).sum():,}")
print(f"P_E: {GLOBAL_P_E:.6f}")
print(f"AUC: {auc_numpy(labels_array, scores_array):.6f}")
print(f"Global threshold: {GLOBAL_THRESHOLD}")
print(f"P_FA at global threshold: {GLOBAL_P_FA:.6f}")
print(f"P_MD at global threshold: {GLOBAL_P_MD:.6f}")
print(f"Detection rate at global threshold: {GLOBAL_DETECTION_RATE:.6f}")

## 3. Per-setting table

`P_E` uses the best threshold for each group. Detection rate uses the one global threshold from Section 2.
This shows the difference between a group score and one operating point used for every group.

In [ ]:
stego_predictions = frame.loc[frame["label"] == 1].copy()
clean_predictions = frame.loc[frame["label"] == 0].copy()
stego_predictions["k_number"] = pd.to_numeric(stego_predictions["k"], errors="raise").astype(int)
stego_predictions["bit_mode_k"] = (
    stego_predictions["bit_mode"].astype(str) + "/" + stego_predictions["k_number"].astype(str)
)

def _placement_value(row: pd.Series) -> str:
    placement = str(row["placement"])
    if placement == "stride":
        return f"stride/{int(float(row['stride']))}"
    return placement

stego_predictions["placement_group"] = stego_predictions.apply(_placement_value, axis=1)
stego_predictions["method_x_placement"] = (
    stego_predictions["method"].astype(str) + " x " + stego_predictions["placement_group"]
)
stego_predictions["bpp_number"] = pd.to_numeric(stego_predictions["bpp_actual"], errors="raise")
BPP_LABELS = ["0.05-0.15", "0.15-0.30", "0.30-0.50"]
stego_predictions["bpp_bin"] = pd.cut(
    stego_predictions["bpp_number"],
    bins=[0.05, 0.15, 0.30, 0.50],
    labels=BPP_LABELS,
    include_lowest=True,
)

def _group_summary(group: pd.DataFrame) -> tuple[int, float, float]:
    cover_ids = set(group["cover_id"])
    paired_clean = clean_predictions.loc[clean_predictions["cover_id"].isin(cover_ids)]
    paired = pd.concat([paired_clean, group], ignore_index=True)
    group_labels = paired["label"].to_numpy(dtype=np.int64)
    group_scores = paired["prob_stego"].to_numpy(dtype=np.float64)
    detection = float((group["prob_stego"].to_numpy(dtype=np.float64) >= GLOBAL_THRESHOLD).mean())
    return len(group), probability_error(group_labels, group_scores), detection

setting_specs = [
    ("bit_mode/k", "bit_mode_k"),
    ("bit_mode", "bit_mode"),
    ("method", "method"),
    ("placement", "placement_group"),
    ("method x placement", "method_x_placement"),
    ("payload_kind", "payload_kind"),
    ("bpp bin", "bpp_bin"),
]
setting_rows = []
for setting_name, setting_column in setting_specs:
    for value, group in stego_predictions.groupby(setting_column, dropna=False, observed=True):
        if pd.isna(value):
            continue
        pairs, p_e, detection = _group_summary(group)
        setting_rows.append({
            "setting": setting_name,
            "value": str(value),
            "pairs": pairs,
            "P_E": p_e,
            "detection_rate_global": detection,
        })

per_setting = pd.DataFrame(
    setting_rows,
    columns=["setting", "value", "pairs", "P_E", "detection_rate_global"],
).sort_values(["setting", "value"]).reset_index(drop=True)
TABLE_PATH = PRED_PATH.with_name(f"{PRED_PATH.stem}_per_setting.csv")
per_setting.to_csv(TABLE_PATH, index=False)
print(per_setting.to_string(index=False))
print(f"Saved per-setting table: {TABLE_PATH}")

## 4. Heatmaps

Each cell shows the group value and pair count. Cells with fewer than 30 pairs are grey.
This avoids strong conclusions from very small groups.

In [ ]:
cell_rows = []
for (bit_mode, k_number, bpp_bin), group in stego_predictions.groupby(
    ["bit_mode", "k_number", "bpp_bin"], dropna=False, observed=True
):
    if pd.isna(bpp_bin):
        continue
    pairs, p_e, detection = _group_summary(group)
    cell_rows.append({
        "bit_mode": str(bit_mode),
        "k": int(k_number),
        "bpp_bin": str(bpp_bin),
        "pairs": pairs,
        "P_E": p_e,
        "detection_rate_global": detection,
    })
cell_table = pd.DataFrame(cell_rows)

def save_heatmap(bit_mode: str, metric: str, file_suffix: str, title: str, vmin: float, vmax: float) -> Path:
    values = np.full((8, len(BPP_LABELS)), np.nan, dtype=np.float64)
    counts = np.zeros((8, len(BPP_LABELS)), dtype=np.int64)
    if not cell_table.empty:
        selected = cell_table.loc[cell_table["bit_mode"] == bit_mode]
        for row in selected.itertuples(index=False):
            row_index = int(row.k) - 1
            column_index = BPP_LABELS.index(row.bpp_bin)
            values[row_index, column_index] = float(getattr(row, metric))
            counts[row_index, column_index] = int(row.pairs)
    visible = np.isfinite(values) & (counts >= 30)
    masked_values = np.ma.masked_where(~visible, values)
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("#d0d0d0")
    fig, axis = plt.subplots(figsize=(8, 7))
    image = axis.imshow(masked_values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set_title(f"{title}: {bit_mode}")
    axis.set_xlabel("bpp bin")
    axis.set_ylabel("k")
    axis.set_xticks(range(len(BPP_LABELS)), BPP_LABELS)
    axis.set_yticks(range(8), range(1, 9))
    for row_index in range(8):
        for column_index in range(len(BPP_LABELS)):
            count = counts[row_index, column_index]
            value = values[row_index, column_index]
            if count == 0:
                text = "n=0"
            elif count < 30:
                text = f"<30\nn={count}"
            else:
                text = f"{value:.3f}\nn={count}"
            axis.text(column_index, row_index, text, ha="center", va="center", fontsize=8)
    fig.colorbar(image, ax=axis, label=metric)
    fig.tight_layout()
    path = PRED_PATH.with_name(f"{PRED_PATH.stem}_{file_suffix}_{bit_mode}.png")
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path

heatmap_paths = []
for bit_mode in ("lowest", "plane"):
    heatmap_paths.append(save_heatmap(bit_mode, "P_E", "heatmap_pe", "P_E", 0.0, 0.5))
for bit_mode in ("lowest", "plane"):
    heatmap_paths.append(save_heatmap(bit_mode, "detection_rate_global", "heatmap_detection", "Detection rate at global threshold", 0.0, 1.0))
for path in heatmap_paths:
    print(f"Saved heatmap: {path}")